In [ ]:
import torch
from torchinfo import summary
# 导入 guided-diffusion 的核心模型和配置
from guided_diffusion.unet import UNetModel
from guided_diffusion.script_util import model_and_diffusion_defaults

def main():
    # --------------------------
    # 1. 配置模型参数（与官方训练/生成一致）
    # --------------------------
    # 获取默认配置（对应 CIFAR-10 32×32 图像，可按需修改）
    defaults = model_and_diffusion_defaults()
    # 自定义关键参数（需与后续输入尺寸匹配）
    model_config = {
        **defaults,
        "image_size": 32,        # 输入图像尺寸（默认 CIFAR-10 为 32×32）
        "in_channels": 3,        # 输入通道数（RGB=3，灰度图=1）
        "out_channels": 3,       # 输出通道数（与输入一致，预测噪声）
        "model_channels": 128,   # U-Net 基础通道数（默认 128，可调整）
        "num_res_blocks": 2,     # 每个 U-Net 块的残差块数量（默认 2）
        "attention_resolutions": (8, 16, 32),  # 哪些分辨率用自注意力（默认）
        "dropout": 0.1,          # dropout 概率（默认）
        "channel_mult": (1, 2, 2, 2),  # 通道数倍增系数（默认，控制编码器/解码器通道变化）
        "conv_resample": True,   # 采样时用卷积（默认）
        "dtype": torch.float32,  # 数据类型（GPU 可改用 torch.float16 混合精度）
        "use_checkpoint": False, # 禁用 checkpoint（避免 torchinfo 解析失败）
        "num_heads": 4,          # 自注意力头数（默认，需与 model_channels 匹配）
        "num_head_channels": None,
        "num_heads_upsample": -1,
        "use_scale_shift_norm": True,  # 论文中的 scale-shift 归一化（默认）
        "resblock_updown": True,       # 残差块内包含采样（默认）
        "use_new_attention_order": False,
    }

    # --------------------------
    # 2. 实例化核心模型（UNetModel）
    # --------------------------
    # 这就是 DDPM 的核心噪声预测网络，也是我们要查看结构的对象
    model = UNetModel(**model_config)
    model.eval()  # 切换评估模式（避免 dropout 等动态层影响结构解析）

    # --------------------------
    # 3. 构造符合要求的输入张量
    # --------------------------
    batch_size = 2  # 批量大小（可任意，不影响结构解析）
    channels = model_config["in_channels"]  # 3
    image_size = model_config["image_size"]  # 32

    # 含噪图像 x_t：形状 (batch_size, channels, image_size, image_size)
    x_t = torch.randn(batch_size, channels, image_size, image_size)
    # 时间步 t：形状 (batch_size,)，取值范围 [0, diffusion_steps-1]（默认 diffusion_steps=1000）
    t = torch.randint(0, 1000, (batch_size,), dtype=torch.long)

    # --------------------------
    # 4. 用 torchinfo 查看模型结构
    # --------------------------
    summary(
        model,
        input_data=(x_t, t),  # 输入为 (含噪图像, 时间步)，与 UNetModel.forward 输入匹配
        col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],
        col_width=15,
        row_settings=["var_names", "depth"],  # 显示变量名和层深度
        dtype=model_config["dtype"],  # 与模型数据类型一致
        device="cpu",  # 先用 CPU 解析（GPU 显存不足时更安全，后续可改 "cuda"）
    )

if __name__ == "__main__":
    main()